Creating the in-memory database, loading the datasets and writing them to memory

In [6]:
import sqlite3
import pandas as pd

# Load datasets from local files
athletes = pd.read_csv("data/athlete_events.csv")
regions = pd.read_csv("data/noc_regions.csv")
tips = pd.read_csv("data/tips.csv")
sales = pd.read_csv("data/chipotle.tsv", sep="\t")

# Connect to SQLite in-memory DB
conn = sqlite3.connect(":memory:")

# Write DataFrames to SQL tables
athletes.to_sql("athletes_table", conn, index=False, if_exists="replace")
regions.to_sql("regions_table", conn, index=False, if_exists="replace")
sales.to_sql("sales_table", conn, index=False, if_exists="replace")

print("Datasets loaded successfully!")

Datasets loaded successfully!


In [11]:
pd.read_sql(
    """SELECT *
    FROM athletes_table
    """,conn
)


,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271111,135569,Andrzej ya,M,29.0,179.0,89.0,Poland-1,POL,1976 Winter,1976,Winter,Innsbruck,Luge,Luge Mixed (Men)'s Doubles,NaN
271112,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Individual",NaN
271113,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Team",NaN
271114,135571,Tomasz Ireneusz ya,M,30.0,185.0,96.0,Poland,POL,1998 Winter,1998,Winter,Nagano,Bobsleigh,Bobsleigh Men's Four,NaN


In [ ]:
#Count the total number of medals won by each country and show the top 5.
pd.read_sql(
    """ SELECT DISTINCT City as City, COUNT(Medal) as Total_Medals
    FROM athletes_table
    WHERE Medal IS NOT "NaN"
    GROUP BY City
    ORDER BY Total_Medals DESC
    LIMIT 5
    """,conn)

,City,Total_Medals
0,London,3624
1,Athina,2602
2,Los Angeles,2123
3,Beijing,2048
4,Rio de Janeiro,2023


In [ ]:
# Calculate the average age of athletes who won a Gold medal
pd.read_sql("""
    SELECT ROUND(AVG(Age), 2) as Average_Age, Medal
            FROM athletes_table
            WHERE Medal = "Gold"

 """,conn)
# Selected all the athletes who got a gold medal and got an avg of their ages

,Average_Age,Medal
0,25.9,Gold


In [20]:
# How many distinct events are there in each sport?
pd.read_sql("""
    SELECT DISTINCT Sport, COUNT(DISTINCT Event) as Distinct_Events
    FROM athletes_table
    GROUP BY Sport
""",conn)

,Sport,Distinct_Events
0,Aeronautics,1
1,Alpine Skiing,10
2,Alpinism,1
3,Archery,29
4,Art Competitions,29
...,...,...
61,Tug-Of-War,1
62,Volleyball,2
63,Water Polo,2
64,Weightlifting,21


In [ ]:
# Find The athletes from USA
pd.read_sql("""
    SELECT DISTINCT Name, NOC
            FROM athletes_table
    WHERE NOC ="USA"
""",conn)

,Name,NOC
0,Per Knut Aaland,USA
1,John Aalberg,USA
2,Stephen Anthony Abas,USA
3,"David ""Dave"" Abbott",USA
4,Jeremy Abbott,USA
...,...,...
9647,Frank Thomas Zuna,USA
9648,David Santos Zuniga,USA
9649,Rami Zur,USA
9650,"Victor Andrew ""Vic"" Zwolak",USA


In [45]:
# Count how many medals were awarded each year.
pd.read_sql(
"""
SELECT   Year, COUNT(*) as Total_Medals
FROM athletes_table
WHERE Medal IS NOT NULL
GROUP BY Year
ORDER BY YEAR
""",conn)

,Year,Total_Medals
0,1896,143
1,1900,604
2,1904,486
3,1906,458
4,1908,831
5,1912,941
6,1920,1308
7,1924,962
8,1928,823
9,1932,739


In [39]:
# Find where weight or height is missing
pd.read_sql("""
SELECT *
FROM athletes_table
WHERE Weight IS NULL OR Height IS NULL

""",conn)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
1,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
2,8,"Cornelia ""Cor"" Aalten (-Strannood)",F,18.0,168.0,NaN,Netherlands,NED,1932 Summer,1932,Summer,Los Angeles,Athletics,Athletics Women's 100 metres,NaN
3,8,"Cornelia ""Cor"" Aalten (-Strannood)",F,18.0,168.0,NaN,Netherlands,NED,1932 Summer,1932,Summer,Los Angeles,Athletics,Athletics Women's 4 x 100 metres Relay,NaN
4,10,"Einar Ferdinand ""Einari"" Aalto",M,26.0,NaN,NaN,Finland,FIN,1952 Summer,1952,Summer,Helsinki,Swimming,Swimming Men's 400 metres Freestyle,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64258,135539,Marius Edmund Zwiller,M,18.0,NaN,NaN,France,FRA,1924 Summer,1924,Summer,Paris,Swimming,Swimming Men's 200 metres Breaststroke,NaN
64259,135542,Werner Zwingli,M,29.0,NaN,NaN,Switzerland,SUI,1956 Winter,1956,Winter,Cortina d'Ampezzo,Cross Country Skiing,Cross Country Skiing Men's 15 kilometres,NaN
64260,135542,Werner Zwingli,M,29.0,NaN,NaN,Switzerland,SUI,1956 Winter,1956,Winter,Cortina d'Ampezzo,Cross Country Skiing,Cross Country Skiing Men's 4 x 10 kilometres R...,NaN
64261,135552,Jan (Johann-) Zybert (Siebert-),M,20.0,NaN,NaN,Poland,POL,1928 Summer,1928,Summer,Amsterdam,Cycling,"Cycling Men's Team Pursuit, 4,000 metres",NaN


In [ ]:
#replacing null height values with avg height using coalesce
pd.read_sql(
"""
SELECT Name, 
    COALESCE(Height,(SELECT AVG(Height) FROM athletes_table))
      AS Height_filled
FROM athletes_table

 """,conn)

,Name,Height_filled
0,A Dijiang,180.00000
1,A Lamusi,170.00000
2,Gunnar Nielsen Aaby,175.33897
3,Edgar Lindenau Aabye,175.33897
4,Christine Jacoba Aaftink,185.00000
...,...,...
271111,Andrzej ya,179.00000
271112,Piotr ya,176.00000
271113,Piotr ya,176.00000
271114,Tomasz Ireneusz ya,185.00000


In [58]:
# sales table
# REturn total sales per item
# First we look up all the table values
pd.read_sql("""
SELECT * 
FROM sales_table
""",conn)
# Then the work
pd.read_sql("""
 SELECT item_name, 
            SUM(quantity * 
            CAST(REPLACE(item_price, '$' , '') AS DECIMAL(10, 2))) as Total_Sales
    FROM sales_table
    GROUP BY item_name
    ORDER BY Total_Sales DESC
""",conn)

,item_name,Total_Sales
0,Chicken Bowl,8044.63
1,Chicken Burrito,6387.06
2,Steak Burrito,4236.13
3,Steak Bowl,2479.81
4,Chips and Guacamole,2475.62
5,Chicken Salad Bowl,1506.25
6,Chicken Soft Tacos,1199.01
7,Chips and Fresh Tomato Salsa,1033.96
8,Veggie Burrito,1002.27
9,Veggie Bowl,901.95


In [59]:
# SELECT top 5 items with highest item price
pd.read_sql("""
SELECT *
FROM sales_table
ORDER BY CAST(REPLACE(item_price, '$' , '') AS DECIMAL(10, 2)) DESC
LIMIT 5
""",conn)

,order_id,quantity,item_name,choice_description,item_price
0,1443,15,Chips and Fresh Tomato Salsa,NaN,$44.25
1,1398,3,Carnitas Bowl,"[Roasted Chili Corn Salsa, [Fajita Vegetables,...",$35.25
2,511,4,Chicken Burrito,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",$35.00
3,1443,4,Chicken Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Chees...",$35.00
4,1443,3,Veggie Burrito,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",$33.75


In [60]:
# How many uniques customer orders are there
pd.read_sql("""
SELECT COUNT(DISTINCT order_id) as Unique_Orders
FROM sales_table
""",conn)

,Unique_Orders
0,1834


In [ ]:
# Find countries with high-performing athletes in the Olympics. 
# Use at least JOIN, NESTED QUERY, CASE, and optionally WITH.
# For each country:Count the number of athletes who won at least one medal.
# Determine the average age of those medalists.
# Create a new column called performance:
# 'High' if average age is below 25
# 'Medium' if between 25 and 30
# 'Low' if above 30

pd.read_sql(
"""
SELECT      r.region AS Country,
            COUNT(DISTINCT m.Name) AS Medalists,
            ROUND(AVG(m.Age),2) AS Average_Age,

        CASE
            WHEN AVG(m.Age) < 25 THEN 'High'
            WHEN AVG(m.Age) BETWEEN 25 AND 30 THEN 'Medium'
            ELSE 'Low'
        END AS Performance

FROM(   SELECT Name,NOC,Age
    FROM athletes_table
    WHERE Medal IS NOT NULL
) m

JOIN regions_table r
ON m.NOC = r.NOC

GROUP BY r.region

ORDER BY Medalists DESC;
""",conn)

,Country,Medalists,Average_Age,Performance
0,USA,3836,24.90,High
1,Russia,2607,25.29,Medium
2,Germany,2563,25.46,Medium
3,UK,1601,27.85,Medium
4,France,1277,27.98,Medium
...,...,...,...,...
131,Curacao,1,19.00,High
132,Botswana,1,18.00,High
133,Bermuda,1,25.00,Medium
134,Barbados,1,24.00,High
